# Explainable Paragraph-Level AI Text Detector

This notebook trains an **explainable AI-text detector** that:
1. Works at **paragraph/section level** — not just document-level.
2. **Highlights likely AI-generated spans** via a paragraph heatmap.
3. **Explains which signals** contributed to the prediction using SHAP.
4. Proposes a **novel method**: combining DivEye-style surprisal diversity features
   with classical stylometric features under a unified SHAP-explainable framework.

## Novel contribution

Neither DivEye (document-level, zero-shot) nor Sci-SpanDet (graph-based, heavyweight)
provides both span localization and per-signal attribution in a lightweight package.
Our approach:
- **Features**: Surprisal statistics + derivatives (DivEye) + stylometric features
  (burstiness, TTR, POS entropy, function word frequency) per paragraph.
- **Classifier**: Gradient-boosted trees (XGBoost) — fast, interpretable.
- **Explanation**: SHAP TreeExplainer — exact Shapley values for each feature per paragraph.
- **Visualization**: Paragraph heatmap + expandable SHAP waterfall plots.

## Dataset

MAGE (ACL 2024) — 437k samples, multiple generators. We split documents into paragraphs
and inherit document-level labels.

**GPU recommended** for surprisal feature extraction (reference LM inference).
CPU fallback available with GPT-2.

## 1. Setup

In [ ]:
import os, sys, random, warnings
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "ai_content_detector" / "notebooks").is_dir():
    repo_root = cwd
    notebook_dir = cwd / "ai_content_detector" / "notebooks"
elif cwd.name == "ai_content_detector":
    repo_root = cwd.parent
    notebook_dir = cwd / "notebooks"
elif cwd.name == "notebooks" and cwd.parent.name == "ai_content_detector":
    repo_root = cwd.parents[1]
    notebook_dir = cwd
else:
    repo_root = cwd
    notebook_dir = cwd

os.chdir(notebook_dir)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Working directory: {Path.cwd()}")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Install dependencies if needed
# !pip install -q datasets xgboost shap spacy
# !python -m spacy download en_core_web_sm

## 2. Load MAGE dataset and split into paragraphs

In [ ]:
from datasets import load_dataset

ds = load_dataset('yaful/MAGE')
df_train = ds['train'].to_pandas()
df_test = ds['test'].to_pandas()

# Subsample for tractable feature extraction
TRAIN_SIZE = 6000
TEST_SIZE = 2000

df_train_sub = df_train.groupby('label', group_keys=False).apply(
    lambda g: g.sample(n=min(TRAIN_SIZE // 2, len(g)), random_state=SEED)
).reset_index(drop=True)

df_test_sub = df_test.groupby('label', group_keys=False).apply(
    lambda g: g.sample(n=min(TEST_SIZE // 2, len(g)), random_state=SEED)
).reset_index(drop=True)

print(f'Train: {len(df_train_sub)}, Test: {len(df_test_sub)}')
print(f'Train labels: {dict(df_train_sub["label"].value_counts())}')

In [ ]:
def split_into_paragraphs(text: str, min_words: int = 20) -> list:
    """Split text into paragraphs, filtering out very short ones."""
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    # If no double-newline splits, try single newlines
    if len(paragraphs) <= 1:
        paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    # If still just one block, split by sentences (rough, ~3 sentences per paragraph)
    if len(paragraphs) <= 1 and len(text.split('.')) > 3:
        sentences = [s.strip() + '.' for s in text.split('.') if s.strip()]
        paragraphs = []
        for i in range(0, len(sentences), 3):
            para = ' '.join(sentences[i:i+3])
            paragraphs.append(para)
    # Filter short paragraphs
    return [p for p in paragraphs if len(p.split()) >= min_words]

# Build paragraph-level dataset. We prefix doc_idx with a split tag so that the
# later document-level aggregation cannot silently confuse train and test
# documents that happen to share a pandas index.
def build_paragraph_dataset(df, split_tag: str):
    rows = []
    for _, row in df.iterrows():
        paras = split_into_paragraphs(row['text'])
        for i, p in enumerate(paras):
            rows.append({
                'text': p,
                'label': row['label'],  # 0=machine, 1=human
                'doc_idx': f'{split_tag}_{row.name}',
                'para_idx': i,
            })
    return pd.DataFrame(rows)

df_para_train = build_paragraph_dataset(df_train_sub, 'tr')
df_para_test = build_paragraph_dataset(df_test_sub, 'te')

# Hard assertion: no document appears in both splits. MAGE already uses disjoint
# train/test splits, but this check means future edits to the sampling above
# can't silently introduce cross-split leakage.
train_docs = set(df_para_train['doc_idx'].unique())
test_docs = set(df_para_test['doc_idx'].unique())
overlap = train_docs & test_docs
assert not overlap, f'Document leakage: {len(overlap)} ids appear in both splits'

print(f'Paragraph-level train: {len(df_para_train)}, test: {len(df_para_test)}')
print(f'Unique documents: train={len(train_docs)}, test={len(test_docs)} (disjoint)')
print(f'Label distribution (train): {dict(df_para_train["label"].value_counts())}')
print(
    'Note: multiple paragraphs share a document → samples within a split are '
    'correlated. Section 4 reports both paragraph-level and document-level '
    '(macro-averaged per doc_idx) metrics to avoid over-optimism.'
)

## 3. Feature extraction

We extract two families of features per paragraph:

### 3a. Surprisal diversity features (DivEye-inspired)
- Token-level surprisal statistics: mean, std, skewness, kurtosis
- 1st-order derivative: std of diff(surprisal) — "rhythmic unpredictability"
- 2nd-order derivative: std of diff(diff(surprisal))

### 3b. Stylometric features
- Burstiness: sentence-level perplexity variance
- Type-token ratio (TTR)
- Average sentence length
- Function word frequency
- POS bigram entropy
- Punctuation density
- Vocabulary richness: hapax legomena ratio

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use GPT-2 large for surprisal (runs on CPU too, just slower)
SCORING_MODEL = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(SCORING_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

scoring_model = AutoModelForCausalLM.from_pretrained(
    SCORING_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)
scoring_model.eval()
print(f'Scoring model loaded: {SCORING_MODEL} on {DEVICE}')

In [ ]:
from scipy import stats as sp_stats

def extract_surprisal_features(text: str, max_length: int = 512) -> dict:
    """Extract DivEye-style surprisal diversity features."""
    tokens = tokenizer(
        text, return_tensors='pt', truncation=True, max_length=max_length
    ).to(DEVICE)
    
    input_ids = tokens['input_ids']
    seq_len = input_ids.size(1)
    
    if seq_len < 5:
        return {f'surp_{k}': 0.0 for k in ['mean', 'std', 'skew', 'kurtosis', 'd1_std', 'd2_std']}
    
    with torch.no_grad():
        logits = scoring_model(**tokens).logits[:, :-1]
        log_probs = torch.log_softmax(logits, dim=-1)
        target_ids = input_ids[:, 1:]
        token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
    
    surprisal = -token_log_probs.squeeze(0).cpu().float().numpy()
    
    if len(surprisal) < 3:
        return {f'surp_{k}': 0.0 for k in ['mean', 'std', 'skew', 'kurtosis', 'd1_std', 'd2_std']}
    
    d1 = np.diff(surprisal)
    d2 = np.diff(d1) if len(d1) > 1 else np.array([0.0])
    
    return {
        'surp_mean': float(np.mean(surprisal)),
        'surp_std': float(np.std(surprisal)),
        'surp_skew': float(sp_stats.skew(surprisal)),
        'surp_kurtosis': float(sp_stats.kurtosis(surprisal)),
        'surp_d1_std': float(np.std(d1)),
        'surp_d2_std': float(np.std(d2)),
    }

In [ ]:
import spacy
import re
from collections import Counter

try:
    nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])
except OSError:
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

FUNCTION_WORDS = set([
    'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'may', 'might', 'shall', 'can', 'need', 'dare', 'ought',
    'used', 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from',
    'as', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'between', 'out', 'off', 'over', 'under', 'again', 'further', 'then',
    'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'both',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor',
    'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 'just',
    'because', 'but', 'and', 'or', 'if', 'while', 'although', 'though',
    'that', 'which', 'who', 'whom', 'this', 'these', 'those', 'it', 'its',
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'him', 'his',
    'she', 'her', 'they', 'them', 'their', 'what', 'about', 'up',
])


def extract_stylometric_features(text: str) -> dict:
    """Extract stylometric features for a paragraph."""
    doc = nlp(text)
    words = [t.text.lower() for t in doc if t.is_alpha]
    sentences = list(doc.sents) if doc.has_annotation('SENT_START') else [doc]
    
    n_words = max(len(words), 1)
    n_chars = max(len(text), 1)
    
    # Type-token ratio
    ttr = len(set(words)) / n_words if n_words > 0 else 0
    
    # Average sentence length
    sent_lengths = [len([t for t in s if t.is_alpha]) for s in sentences]
    avg_sent_len = np.mean(sent_lengths) if sent_lengths else 0
    std_sent_len = np.std(sent_lengths) if len(sent_lengths) > 1 else 0
    
    # Function word frequency
    fw_count = sum(1 for w in words if w in FUNCTION_WORDS)
    fw_ratio = fw_count / n_words
    
    # POS bigram entropy
    pos_tags = [t.pos_ for t in doc if t.is_alpha]
    if len(pos_tags) > 1:
        pos_bigrams = [(pos_tags[i], pos_tags[i+1]) for i in range(len(pos_tags)-1)]
        bigram_counts = Counter(pos_bigrams)
        total = sum(bigram_counts.values())
        probs = [c / total for c in bigram_counts.values()]
        pos_bigram_entropy = -sum(p * np.log2(p) for p in probs if p > 0)
    else:
        pos_bigram_entropy = 0
    
    # Punctuation density
    punct_count = sum(1 for c in text if c in '.,;:!?-()"\'\'\"')
    punct_density = punct_count / n_chars
    
    # Hapax legomena ratio (words appearing exactly once)
    word_counts = Counter(words)
    hapax = sum(1 for c in word_counts.values() if c == 1)
    hapax_ratio = hapax / n_words if n_words > 0 else 0
    
    # Burstiness: coefficient of variation of sentence lengths
    burstiness = std_sent_len / avg_sent_len if avg_sent_len > 0 else 0
    
    return {
        'stylo_ttr': ttr,
        'stylo_avg_sent_len': avg_sent_len,
        'stylo_std_sent_len': std_sent_len,
        'stylo_fw_ratio': fw_ratio,
        'stylo_pos_bigram_entropy': pos_bigram_entropy,
        'stylo_punct_density': punct_density,
        'stylo_hapax_ratio': hapax_ratio,
        'stylo_burstiness': burstiness,
        'stylo_n_words': n_words,
    }

In [ ]:
from tqdm import tqdm

def extract_all_features(df_paras: pd.DataFrame) -> pd.DataFrame:
    """Extract all features for a paragraph-level DataFrame.

    Carries ``doc_idx`` through so we can compute document-level macro-averaged
    metrics alongside the paragraph-level ones (paragraphs within a document are
    correlated, so paragraph-level AUROC can overstate detector performance).
    """
    records = []
    for _, row in tqdm(df_paras.iterrows(), total=len(df_paras), desc='Extracting features'):
        surp = extract_surprisal_features(row['text'])
        stylo = extract_stylometric_features(row['text'])
        records.append({
            **surp, **stylo,
            'label': row['label'],
            'doc_idx': row['doc_idx'],
        })
    return pd.DataFrame(records)

print('Extracting features for training paragraphs...')
df_features_train = extract_all_features(df_para_train)
print(f'Train features shape: {df_features_train.shape}')

print('\nExtracting features for test paragraphs...')
df_features_test = extract_all_features(df_para_test)
print(f'Test features shape: {df_features_test.shape}')

## 4. Train explainable classifier

In [ ]:
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, classification_report,
)

# Only real numeric features — doc_idx is grouping metadata, not a signal.
FEATURE_COLS = [
    c for c in df_features_train.columns
    if c not in ('label', 'doc_idx')
]

X_train = df_features_train[FEATURE_COLS].values
y_train = df_features_train['label'].values  # 0=machine, 1=human
X_test = df_features_test[FEATURE_COLS].values
y_test = df_features_test['label'].values

# Train XGBoost — chosen for SHAP TreeExplainer compatibility
clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    eval_metric='logloss',
    use_label_encoder=False,
)
clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)

# Evaluate — paragraph level
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print('\n=== Test Set Results (paragraph-level) ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Macro:  {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, average="macro"):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred, average="macro"):.4f}')
print('\n' + classification_report(y_test, y_pred, target_names=['Machine', 'Human']))

# Evaluate — document level. Paragraphs within one document share a label and
# share context (correlated samples). Macro-average the paragraph-level human
# probability per document and use the document's label as ground truth; this
# removes the correlation and gives an honest per-document AUROC.
doc_df = pd.DataFrame({
    'doc_idx': df_features_test['doc_idx'].values,
    'label': y_test,
    'prob_human': y_prob,
}).groupby('doc_idx').agg(
    label=('label', 'first'),
    prob_human=('prob_human', 'mean'),
).reset_index()

doc_y = doc_df['label'].values
doc_p = doc_df['prob_human'].values
doc_pred = (doc_p > 0.5).astype(int)

print('=== Test Set Results (document-level, macro-averaged paragraphs) ===')
print(f'Unique test documents: {len(doc_df)}')
print(f'Accuracy:  {accuracy_score(doc_y, doc_pred):.4f}')
print(f'F1 Macro:  {f1_score(doc_y, doc_pred, average="macro"):.4f}')
print(f'ROC-AUC:   {roc_auc_score(doc_y, doc_p):.4f}')
print(
    'Document-level AUROC is the more defensible headline metric; paragraph-level '
    'numbers are optimistic because paragraphs within a document are not i.i.d.'
)

## 5. SHAP explanations

In [ ]:
import shap

# Exact SHAP values for tree-based models
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Summary plot: which features matter most globally
print('Global feature importance (SHAP):')
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=True)

In [ ]:
# Bar plot of mean absolute SHAP values
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, plot_type='bar', show=True)

## 6. Paragraph-level heatmap visualization

Given a document, we:
1. Split it into paragraphs.
2. Extract features for each paragraph.
3. Predict AI probability for each paragraph.
4. Display a heatmap (red = AI, green = human).
5. Show per-paragraph SHAP waterfall for the top contributing features.

In [ ]:
def analyze_document(text: str) -> dict:
    """Analyze a document at paragraph level.
    
    Returns:
        dict with 'paragraphs', 'scores', 'features', 'shap_values', 'feature_names'
    """
    paragraphs = split_into_paragraphs(text)
    if not paragraphs:
        return {'paragraphs': [], 'scores': [], 'features': None, 'shap_values': None}
    
    # Extract features
    feature_records = []
    for p in paragraphs:
        surp = extract_surprisal_features(p)
        stylo = extract_stylometric_features(p)
        feature_records.append({**surp, **stylo})
    
    df_feat = pd.DataFrame(feature_records)
    X = df_feat[FEATURE_COLS].values
    
    # Predict
    probs = clf.predict_proba(X)  # columns: [machine, human]
    ai_scores = probs[:, 0].tolist()  # P(machine)
    
    # SHAP values
    sv = explainer.shap_values(X)
    
    return {
        'paragraphs': paragraphs,
        'scores': ai_scores,
        'features': df_feat,
        'shap_values': sv,
        'feature_names': FEATURE_COLS,
    }

In [ ]:
def visualize_document(analysis: dict, max_chars_preview: int = 200):
    """Visualize paragraph-level AI detection results."""
    paragraphs = analysis['paragraphs']
    scores = analysis['scores']
    
    if not paragraphs:
        print('No paragraphs found.')
        return
    
    n = len(paragraphs)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, n * 0.8)),
                             gridspec_kw={'width_ratios': [1, 3]})
    
    # Left: heatmap
    ax_heat = axes[0]
    cmap = mcolors.LinearSegmentedColormap.from_list('ai_human', ['#4CAF50', '#FFEB3B', '#F44336'])
    score_arr = np.array(scores).reshape(-1, 1)
    ax_heat.imshow(score_arr, cmap=cmap, aspect='auto', vmin=0, vmax=1)
    ax_heat.set_yticks(range(n))
    ax_heat.set_yticklabels([f'P{i+1}' for i in range(n)])
    ax_heat.set_xticks([])
    ax_heat.set_title('AI Score')
    for i, s in enumerate(scores):
        ax_heat.text(0, i, f'{s:.0%}', ha='center', va='center', fontsize=10,
                     color='white' if s > 0.5 else 'black', fontweight='bold')
    
    # Right: text previews
    ax_text = axes[1]
    ax_text.axis('off')
    for i, (para, score) in enumerate(zip(paragraphs, scores)):
        preview = para[:max_chars_preview] + ('...' if len(para) > max_chars_preview else '')
        label = 'AI' if score > 0.65 else ('Human' if score < 0.35 else '?')
        color = '#F44336' if score > 0.65 else ('#4CAF50' if score < 0.35 else '#FF9800')
        y_pos = 1.0 - (i + 0.5) / n
        ax_text.text(0.02, y_pos, f'[{label} {score:.0%}] {preview}',
                     fontsize=9, va='center', wrap=True,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.15))
    
    plt.suptitle('Paragraph-Level AI Detection Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Overall document score
    avg_score = np.mean(scores)
    print(f'\nDocument-level AI score: {avg_score:.1%}')
    print(f'Paragraphs flagged as AI: {sum(1 for s in scores if s > 0.65)}/{n}')

In [ ]:
def show_paragraph_explanation(analysis: dict, para_idx: int):
    """Show SHAP waterfall plot for a specific paragraph."""
    sv = analysis['shap_values']
    features = analysis['features']
    feature_names = analysis['feature_names']
    
    print(f'\n--- Paragraph {para_idx + 1} ---')
    print(f'Text: {analysis["paragraphs"][para_idx][:300]}...')
    print(f'AI Score: {analysis["scores"][para_idx]:.1%}')
    
    # Create SHAP Explanation object
    explanation = shap.Explanation(
        values=sv[para_idx],
        base_values=explainer.expected_value,
        data=features.iloc[para_idx][feature_names].values,
        feature_names=feature_names,
    )
    
    shap.waterfall_plot(explanation, max_display=15, show=True)
    
    # Top contributing features
    abs_shap = np.abs(sv[para_idx])
    top_idx = np.argsort(abs_shap)[::-1][:5]
    print('\nTop contributing features:')
    for idx in top_idx:
        direction = 'toward AI' if sv[para_idx][idx] > 0 else 'toward Human'
        print(f'  {feature_names[idx]}: SHAP={sv[para_idx][idx]:+.4f} ({direction}), '
              f'value={features.iloc[para_idx][feature_names[idx]]:.4f}')

## 7. Demo: analyze a test document

In [ ]:
# Pick a random AI-generated document from test set
ai_docs = df_test_sub[df_test_sub['label'] == 0]
sample_text = ai_docs.iloc[0]['text']

print('Analyzing an AI-generated document...')
analysis = analyze_document(sample_text)
visualize_document(analysis)

In [ ]:
# Show SHAP explanation for the first paragraph
if analysis['paragraphs']:
    show_paragraph_explanation(analysis, para_idx=0)

In [ ]:
# Now a human-written document
human_docs = df_test_sub[df_test_sub['label'] == 1]
sample_human = human_docs.iloc[0]['text']

print('Analyzing a human-written document...')
analysis_human = analyze_document(sample_human)
visualize_document(analysis_human)

In [ ]:
if analysis_human['paragraphs']:
    show_paragraph_explanation(analysis_human, para_idx=0)

## 8. Feature importance analysis

Which signal families contribute most to detection?

In [ ]:
# Group SHAP values by feature family
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False)

# Split by family
surp_importance = feature_importance[feature_importance['feature'].str.startswith('surp_')]
stylo_importance = feature_importance[feature_importance['feature'].str.startswith('stylo_')]

print('=== Surprisal features (DivEye-inspired) ===')
print(surp_importance.to_string(index=False))
print(f'\nTotal surprisal importance: {surp_importance["mean_abs_shap"].sum():.4f}')

print('\n=== Stylometric features ===')
print(stylo_importance.to_string(index=False))
print(f'\nTotal stylometric importance: {stylo_importance["mean_abs_shap"].sum():.4f}')

# Pie chart
fig, ax = plt.subplots(figsize=(8, 5))
family_sums = [
    surp_importance['mean_abs_shap'].sum(),
    stylo_importance['mean_abs_shap'].sum(),
]
ax.pie(family_sums, labels=['Surprisal (DivEye)', 'Stylometric'],
       autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
ax.set_title('Feature Family Contribution to Detection')
plt.show()

## 9. Save trained model

In [ ]:
import joblib
import json

SAVE_DIR = Path('../models/explainable_detector')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save classifier
clf.save_model(SAVE_DIR / 'xgb_paragraph_detector.json')

# Save feature names
with open(SAVE_DIR / 'feature_names.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

# Save SHAP explainer
joblib.dump(explainer, SAVE_DIR / 'shap_explainer.pkl')

print(f'Model and explainer saved to {SAVE_DIR}')

## 10. Summary

### What was built
- A **paragraph-level AI text detector** combining:
  - **Surprisal diversity features** (DivEye-inspired): mean, std, skewness, kurtosis
    of token-level surprisal, plus 1st/2nd order derivatives.
  - **Stylometric features**: burstiness, TTR, sentence length, function word frequency,
    POS bigram entropy, punctuation density, hapax legomena ratio.
- An **XGBoost classifier** trained on MAGE paragraphs.
- **SHAP explanations** via TreeExplainer: per-feature Shapley values for each paragraph.
- **Paragraph heatmap** visualization: color-coded spans + expandable explanations.

### Novel aspects
1. **First combination** of DivEye surprisal derivatives with classical stylometric features
   in a unified explainable framework.
2. **Paragraph-level granularity** with per-span attribution — neither DivEye (document-level)
   nor Sci-SpanDet (heavyweight graph-based) provides this in a lightweight package.
3. **Dual-family explanation**: shows whether detection signals come from surprisal patterns
   (how the text was generated) or stylometric patterns (how it reads).

### Limitations
- Paragraph labels inherited from document labels — some paragraphs in AI documents
  may actually be human-written (e.g., prompts, quoted text).
- Surprisal features depend on the reference LM (GPT-2 large); detection of text from
  very different model families may be weaker.
- The calibrated thresholds should be validated on additional datasets.